# Introduction: PhysicsNeMo concepts

[Start Here](Start_Here.ipynb) · Previous: [Setup](00_Setup.ipynb) · Next: [Lab 1: PINN fundamentals](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem)

A physics-informed neural network predicts a solution and is trained to satisfy an equation and its initial or boundary conditions. Start with those ingredients here, then train a model in [Lab 1](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem).

This notebook is for reading; the executable exercises begin in Lab 1.

## Introduction to PhysicsNeMo

PhysicsNeMo provides PyTorch models and tools for physics-based training. This course uses version 2.2.2: fully connected networks for PINNs, FNO and AFNO for neural operators, and symbolic equations for physics losses.

[Official overview](https://docs.nvidia.com/physicsnemo/latest/index.html) · [Version 2.2.2 source](https://github.com/NVIDIA/physicsnemo/tree/v2.2.2) · [Version 2.0 migration guide](https://github.com/NVIDIA/physicsnemo/blob/v2.2.2/v2.0-MIGRATION-GUIDE.md)

### Components used in this course

| Component | Role | Example in this course |
|---|---|---|
| `physicsnemo.models` | PyTorch model architectures | `FullyConnected`, followed by FNO and AFNO |
| SymPy | Describe fields and derivatives symbolically | `Function("u")(x, y, t)`, `u.diff(x, 2)` |
| `physicsnemo.sym` | Hold equation definitions and evaluate residual tensors | `PDE`, `PhysicsInformer` |
| Course helpers in `ETC/runtime` | Connect lesson settings and tensors to the library | `create_model`, `create_informer`, `residuals` in the PINN challenges |
| Data and coordinate sampling | Prepare training inputs | `torch.rand`, initial-condition data |
| Explicit training loop | Evaluate losses and optimize parameters | `torch.optim.Adam` or `torch.optim.LBFGS`, `backward`, `step` |
| Validation and inference | Check equations, boundary conditions, and prediction error | Fixed evaluation coordinates; Lab 4 forecasts compared with ERA5 and persistence |

Sym stands for Symbolic. SymPy describes the equation; `PhysicsInformer` evaluates it on predicted fields using PyTorch tensors. The introductory MLP exercises also run on CPU. With a compatible CUDA-enabled PyTorch environment, select `--device cuda` to use a GPU.

### Read through the course helpers

`create_model`, `create_informer`, and `residuals` are course utilities in [ETC/runtime/pinn.py](ETC/runtime/pinn.py), not public PhysicsNeMo APIs. `create_model` constructs `physicsnemo.models.mlp.FullyConnected`; `create_informer` configures `physicsnemo.sym.eq.phy_informer.PhysicsInformer` through a course adapter; `residuals` evaluates the network, prepares named tensors and calls `informer.forward`. The model predicts field values; the informer returns equation residuals; the training loop reduces them to a loss.

Labs 1–3 use related helpers such as `mlp` and `informer` in [ETC/runtime/labs.py](ETC/runtime/labs.py), with different arguments and lesson-specific model code. Lab 4 loads a pretrained AFNO model for weather inference; it does not use a PINN training loop. Follow each file's imports rather than assuming every lesson uses the challenge helper signatures.

For a complete worked path from `student_equations` to `total.backward()`, read [From course helpers to PhysicsNeMo](ETC/course_materials/PHYSICSNEMO_WORKFLOW.md). It follows Wave Level 1 and explains the operator workflow used later in Challenge 4.

### How the PINN lessons train a model

1. Specify the coordinates, predicted quantities, domain and units.
2. Write SymPy expressions in a PhysicsNeMo `PDE` subclass and create a `FullyConnected` network.
3. Sample coordinates and calculate losses for the equation, conditions and any observations.
4. Evaluate on a fixed grid and check which coordinates or parameter values overlap training.
5. Check the lesson's step count, batch size, learning rate and network size in its Python defaults or YAML configuration.
6. Train with the lesson's optimizer: L-BFGS in Lab 1, Adam in Lab 2, and Adam followed by L-BFGS in Lab 3. Compare predictions, residuals and analytical solutions where available. Lab 4 has no optimizer: it runs a pretrained weather model.

Labs 1–3 use explicit PyTorch training loops and lesson-specific settings. This matches the [official PhysicsNeMo physics-guided workflow](https://docs.nvidia.com/physicsnemo/latest/user-guide/physics_addition.html): PhysicsNeMo supplies models and physics tools, while your loop controls sampling, losses and optimization. You can inspect those decisions in the linked Python files.

## PINNs, neural operators and PINO

A data-driven model learns from input and target pairs. In Challenge 4, FNO and AFNO take a forcing field and predict its solution field. A PINN instead takes coordinates and learns a solution using equation and condition losses, sometimes with observations. PINO adds a physics residual to neural-operator training.

In [Lab 4: AI Weather Forecasting with FourCastNet](01_labs/04_weather_forecasting/Lab_4_Weather_Forecasting.ipynb), an already-trained AFNO maps 26 global weather fields to the same fields six hours later. Reusing each prediction as the next input produces a 48-hour forecast. Only the initial ERA5 state enters forecasting; later ERA5 reanalysis is reserved for verification. Compare the forecast with a persistence baseline that keeps the initial weather unchanged. ERA5 combines observations and a numerical model; it is not direct observation alone.

This is pretrained inference, not training from scratch or minimizing PDE residuals. A single historical weather case illustrates the workflow, not general forecast reliability. The previous Navier–Stokes Lab is retained as [reference material](ETC/reference_labs/04_navier_stokes/Lab_4_Navier_Stokes.ipynb).

`FullyConnected`, FNO and AFNO describe model architectures. PINN and PINO describe how physical information enters training. The same model can be trained with different losses; check both its prediction error and the equations it is meant to satisfy.

Challenge 4 uses the public `physicsnemo.models.fno.FNO` and `physicsnemo.models.afno.AFNO` classes. Its PINO level adds `PhysicsInformer` with `grad_method="spectral"` on periodic grids. Data loss compares normalized fields; the PDE residual is evaluated after returning the prediction and forcing to their physical scale.

### A network for the first problem

```python
from physicsnemo.models.mlp import FullyConnected
model = FullyConnected(in_features=1, out_features=1, layer_size=64,
                       num_layers=3, activation_fn="tanh")
```

The next lab connects this model to a PDE and a training loop. Smooth activations such as `tanh` are suitable for PINNs that require second derivatives.

### Two kinds of derivative

The network takes coordinates $x$ and predicts $u_\theta(x)$. `PhysicsInformer` differentiates with respect to the spatial coordinate to form the residual $u_{\theta,xx}-1$. The loss combines this residual with the errors at the two boundaries.

`loss.backward()` then differentiates the loss with respect to the network parameters $\theta$; `optimizer.step()` updates those parameters. The coordinates are inputs, not quantities the optimizer changes.

For time-dependent PINN challenges, the course `residuals` helper computes `u__t` and `u__t__t` with `torch.autograd.grad` and supplies those tensors explicitly; `PhysicsInformer` handles the spatial derivatives. A residual measures equation mismatch, not solution error: the zero field satisfies the homogeneous wave PDE but fails Wave Level 1's nonzero initial conditions.

You can follow these operations in `BasicPINN`, `loss_terms`, and `optimize_lab` in [pinn_basics.py](01_labs/01_pinn/source_code/pinn_basics.py). Its L-BFGS update uses `optimizer.step(closure)`; the closure recomputes the loss and gradients on fixed training points. For $u''=1$ with zero values at both ends, should the curve lie above or below zero? The analytical solution is used for comparison, not as a training target.

### Next: Lab 1

Open [PINN fundamentals](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem) and run the first example.

[Start Here](Start_Here.ipynb) · Next: [Lab 1: PINN fundamentals](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem)

--- 

Further resources: [Open Hackathons](https://www.openhackathons.org/s/technical-resources) and the [OpenACC and Hackathons Slack channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.